# Сохранённые результаты: все модели, графики и риск

Выполните **Run All / Выполнить все ячейки**. `RESULT`, `LIVE` и открытый старый
ноутбук не требуются. Обучение регрессоров и прерванный `forecast` не запускаются.
Используются уже сохранённые ошибки, пороги и признаки `conditions` и `related`:
это сохраняет именно оценки тех моделей, которые работали в соответствующие месяцы.
Повторно прогонять последнюю модель по её прошлому было бы некорректно.

Ноутбук рассчитан и на предыдущую версию проекта с LiveReport. Положите его
в `training`, рядом с `02_Обучение.ipynb`, и откройте на компьютере с результатами
и исходной телеметрией. Остановите старую ячейку обучения перед запуском этого ноутбука.
Телеметрия нужна для режимов, реестра остановов и исходов при обучении риска.
**Одной папки весов без сохранённых отчётов и истории недостаточно.**

Настройки автоматически читаются из **сохранённого на диске** `02_Обучение.ipynb`.
Сначала сохраните старый ноутбук (Ctrl+S). Если его параметры не сохранены либо он
называется иначе, поправьте настройки ниже. Главный параметр — прежний `output`.

Риск обучается отдельно для каждой схемы входов, с объединением доступных котлов
одной группы при `risk_policy.scope="group"`. Для графиков выполняется помесячная
проверка без знания будущих исходов; финальная рабочая модель риска сохраняется
отдельно. Если независимых аварий недостаточно, появится причина, а не выдуманный процент.


In [ ]:
from pathlib import Path
import ast, inspect, json, os, sys, time, traceback
import numpy as np
import pandas as pd
from IPython.display import display, Markdown

TRAINING_DIR = next((p.resolve() for p in (Path.cwd(), Path.cwd()/"training")
                     if (p/"lib/monitoring_v3").is_dir()), None)
if TRAINING_DIR is None:
    raise FileNotFoundError("Откройте ноутбук из папки training или корня проекта.")
os.chdir(TRAINING_DIR)
for p in (TRAINING_DIR, TRAINING_DIR/"lib"):
    if str(p) not in sys.path:
        sys.path.insert(0, str(p))

from notebook_reports import notebook_config, experiment_directories, show_table, read_report
from monitoring_v3.data import load_sources
from monitoring_v3.progress import Progress
from monitoring_v3.threads import numerical_threads
from monitoring_v3.risk_history import build_risk_history, read_risk_timeline
from monitoring_v3.training_plot import plot_training_timeline
from monitoring_v3.risk_reporting import readiness_rows
from interpretation_training import train_risk_group
import matplotlib.pyplot as plt
print("Папка проекта:", TRAINING_DIR)


## 1. Настройки

`OUTPUT=None` — взять прежний `output` из настроек обучения. Если результаты лежат
в другом месте, укажите, например, `OUTPUT="runs/incremental_validation"` или полный путь.
`SETTINGS_OVERRIDE` — только необходимые исправления путей/параметров прежнего запуска.
Не меняйте пороги работы, правила датчиков и разметку без необходимости: сохранённые
отклонения рассчитаны с прежними настройками.

`UNITS=None` — все доступные котлы; `EXPERIMENTS=None` — все подходящие эксперименты.
Чтобы посмотреть один котёл, задайте `UNITS=["К-5"]`.
`VIEWS` — пять основных схем тревог; `None` — все сохранённые схемы, включая группы датчиков.
`BUDGET=None` — основной сохранённый бюджет; другие значения должны быть уже рассчитаны.
`START/END` обрезают только графики, не историю обучения риска.
`TRAIN_RISK=False` — только просмотр ранее сохранённых оценок.
`AS_OF=None` — обучение до конца доступных сохранённых признаков; можно указать дату.
Рабочие классификаторы и история сохраняются, регрессоры не меняются.


In [ ]:
SOURCE_NOTEBOOK = "02_Обучение.ipynb"
OUTPUT = None
SETTINGS_OVERRIDE = {}
# Например: {"data_paths": {"К-5": "D:/private/К-5/hourly.parquet"}}

VARIANTS = ["conditions", "related"]
METHODS = None                    # None — все; например ["CatBoost"]
UNITS = None
EXPERIMENTS = None
VIEWS = ["mean::instant", "top5::instant", "max::instant",
         "fraction::instant", "mean::slow48"]
BUDGET = None
START = None
END = None
TRAIN_RISK = True
AS_OF = None


In [ ]:
def saved_literals(path):
    values = {}
    if not path.exists():
        return values
    notebook = json.loads(path.read_text(encoding="utf-8-sig"))
    for cell in notebook.get("cells", []):
        if cell.get("cell_type") != "code":
            continue
        try:
            tree = ast.parse("".join(cell["source"]))
        except SyntaxError:
            continue
        for statement in tree.body:
            if not isinstance(statement, ast.Assign):
                continue
            for target in statement.targets:
                if isinstance(target, ast.Name) and target.id in ("SETTINGS", "SENSOR_RULES"):
                    try:
                        values[target.id] = ast.literal_eval(statement.value)
                    except (ValueError, TypeError):
                        raise ValueError(f"{target.id} в {path.name} содержит выражения. "
                                         "Укажите эти параметры явно в SETTINGS_OVERRIDE.") from None
    return values

previous = saved_literals(TRAINING_DIR/SOURCE_NOTEBOOK)
settings = {**previous.get("SETTINGS", {}), **SETTINGS_OVERRIDE}
if OUTPUT is not None:
    settings["output"] = str(OUTPUT)
CFG = notebook_config(settings, previous.get("SENSOR_RULES", {}))
ROOT = Path(CFG["output"]).resolve()
print("Читаем результаты из:", ROOT)
ITEMS = [i for i in experiment_directories(CFG)
         if i["variant"] in VARIANTS
         and (METHODS is None or i["method"] in METHODS)
         and (UNITS is None or i["unit"] in UNITS)
         and (EXPERIMENTS is None or i["experiment"] in EXPERIMENTS)]
if not ITEMS:
    raise FileNotFoundError("В выбранной output нет сохранённых результатов для этих фильтров. "
                            "Укажите OUTPUT из старого обучения. forecast продолжать не нужно.")
REPORT_DIR = ROOT/"saved_results_report"
REPORT_DIR.mkdir(parents=True, exist_ok=True)
catalog = []
for item in ITEMS:
    p = item["directory"]/"prequential_errors.parquet"
    e = pd.read_parquet(p)
    catalog.append({**{k:v for k,v in item.items() if k != "directory"},
                    "начало оценок": e.index.min(), "конец оценок": e.index.max(),
                    "строк": len(e),
                    "признаки риска": (item["directory"]/"interpretation_features.parquet").exists()})
CATALOG = pd.DataFrame(catalog)
show_table("Найденные сохранённые результаты", CATALOG, True)
print("Если котла здесь нет, до него мог не дойти прерванный запуск.")
print("Графики и таблицы будут сохранены в:", REPORT_DIR)


## 2. Сводка завершённых расчётов

Читаются файлы на диске; переменная `RESULT` не нужна. Сводка может не включать
прерванный эксперимент, даже если часть его месячных файлов успела сохраниться.
Глобальный `status.csv` после прерывания не используется как признак завершения.


In [ ]:
for filename, title in [
    ("comparison.csv", "Качество завершённых моделей и схем тревог"),
    ("comparison_periods.csv", "Качество по периодам"),
    ("models.csv", "Сохранённые модели для дашборда"),
]:
    path = ROOT/filename
    if path.exists():
        table = read_report(path)
        if "experiment" in table:
            keys = CATALOG[["experiment", "unit"]].drop_duplicates()
            table = table.merge(keys, on=["experiment", "unit"] if "unit" in table else ["experiment"])
        show_table(title, table, filename == "comparison.csv")


## 3. Подготовка истории для риска и графиков

Телеметрия читается один раз, регрессоры не обучаются. Используются прежние
настройки подготовки. Исходы риска строятся автоматически из подтверждённых
аварийных остановов; неизвестные остановы не превращаются в «норму».


In [ ]:
needed = {i["unit"] for i in ITEMS}
missing_features = [str(i["directory"]) for i in ITEMS
                    if not (i["directory"]/"interpretation_features.parquet").exists()]
if TRAIN_RISK and missing_features:
    raise FileNotFoundError("Нет сохранённых признаков для риска: " + "; ".join(missing_features)
                            + ". Для просмотра без обучения задайте TRAIN_RISK=False.")
# Убираем только котлы без выбранных сохранённых результатов.
CFG["groups"] = {g: [u for u in units if u in needed]
                 for g, units in CFG["groups"].items() if needed.intersection(units)}
covered = {u for units in CFG["groups"].values() for u in units}
if needed-covered:
    raise ValueError("Добавьте в SETTINGS_OVERRIDE['groups'] котлы: " + str(sorted(needed-covered)))
with numerical_threads(CFG["threads"]):
    CONTEXTS, ISSUES = load_sources(CFG, REPORT_DIR/"audit", strict=False)
if ISSUES:
    show_table("Проблемы источников", pd.DataFrame(ISSUES), True)
if needed-set(CONTEXTS):
    raise ValueError("Недоступна телеметрия котлов: " + str(sorted(needed-set(CONTEXTS)))
                     + ". Исправьте пути в SETTINGS_OVERRIDE.")
print("Подготовлена история:", ", ".join(sorted(CONTEXTS)))


## 4. Обучение риска и графики для каждой модели

Сначала для группы считается история риска по месяцам, затем обучается рабочая
голова на доступных к `AS_OF` исходах. На график попадают только помесячные оценки,
а не предсказания последней модели на её собственном обучении. `conditions` и
`related` имеют разные признаки и отдельные классификаторы.

После каждой готовой группы сразу появляются её графики. Каждая картинка и таблица
также сохраняется в `saved_results_report`. Если риск пока обучить нельзя,
графики отклонений всё равно выводятся с причиной отсутствия оценки.
Повторный запуск использует кеш истории. Ячейку можно прервать и повторить;
уже записанные PNG/CSV останутся. Ошибка отдельной группы печатается с traceback,
остальные группы продолжаются.

**Проценты** — откалиброванная вероятность. **Балл** — оценка модели без достаточной
калибровки, это не вероятность. Подпись `при подтверждении → максимум` показывает
оценку в начале тревоги и её последующее развитие. Максимум не был известен заранее.


In [ ]:
families = {}
for item in ITEMS:
    spec = json.loads((item["directory"].parent/"experiment.json").read_text(encoding="utf-8"))
    # Разные сценарии, схемы входов и источники transfer не смешиваются.
    key = (spec.get("group", ""), item["method"], item["variant"],
           spec.get("horizon", 0), item["scenario"],
           tuple(spec.get("sources", [])) if item["scenario"] == "transfer" else ())
    if CFG["risk_policy"]["scope"] != "group":
        key += (item["experiment"], item["unit"])
    families.setdefault(key, []).append(item)

RISK_ROWS, FILES, FAILURES = [], [], []
for number, (family, members) in enumerate(families.items(), 1):
    display(Markdown(f"### {number}/{len(families)} · {family[1]} · {family[2]} · "
                     + ", ".join(sorted({i["unit"] for i in members}))))
    progress = Progress(REPORT_DIR/"logs"/str(number))
    statuses = {}
    if TRAIN_RISK:
        try:
            dates = [pd.read_parquet(i["directory"]/"interpretation_features.parquet").index.max()
                     for i in members]
            dates = [pd.Timestamp(t) for t in dates if pd.notna(t)]
            if not dates:
                raise ValueError("Сохранённые признаки риска пусты.")
            date = min(pd.Timestamp(AS_OF), max(dates)) if AS_OF is not None else max(dates)
            with numerical_threads(CFG["threads"]):
                with progress.stage("Помесячные оценки риска без знания будущего"):
                    build_risk_history(CFG, members, CONTEXTS, date, progress,
                        on_update=lambda done,total: progress.set_detail(f"Срезы риска: {done}/{total}"))
                with progress.stage("Сохранение рабочих классификаторов риска"):
                    statuses = train_risk_group(CFG, members, CONTEXTS, date)
            for item in members:
                path = item["directory"]/"interpretation_status.json"
                previous_status = json.loads(path.read_text(encoding="utf-8")) if path.exists() else {}
                previous_status.update(statuses)
                path.write_text(json.dumps(previous_status, ensure_ascii=False, indent=2, default=str), encoding="utf-8")
            progress.say("Риск рассчитан; построение графиков", status="complete")
        except Exception as exc:
            FAILURES.append({"семейство": str(family), "этап": "риск", "ошибка": repr(exc)})
            traceback.print_exc()
            print("Переходим к сохранённым графикам; отсутствующие оценки риска не подменяются.")
    for item in members:
        directory, unit = item["directory"], item["unit"]
        label = item["experiment"] + " · " + unit
        folder = REPORT_DIR/item["experiment"]/unit
        folder.mkdir(parents=True, exist_ok=True)
        status_path = directory/"interpretation_status.json"
        item_status = json.loads(status_path.read_text(encoding="utf-8")) if status_path.exists() else statuses
        ready = readiness_rows(item_status, CFG)
        if len(ready):
            show_table(label + " · готовность классификаторов", ready, True)
            ready.to_csv(folder/"classifier_readiness.csv", index=False, encoding="utf-8-sig")
        for head, status in item_status.items():
            if head.startswith("risk_"):
                RISK_ROWS.append(dict(experiment=item["experiment"], unit=unit, head=head, **status))
        risk = read_risk_timeline(directory, CFG, CONTEXTS[unit])
        if risk is None:
            print(label + ": совместимой помесячной истории риска нет; графики будут без оценок риска.")
        elif len(risk):
            show_table(label + " · доступность исторических оценок",
                risk.groupby(["horizon_hours", "status"], dropna=False).size().reset_index(name="часов"))
        budget = float(CFG["alarm_budget"] if BUDGET is None else BUDGET)
        suffix = "__" + str(budget)
        available = {}
        for path in directory.glob("*.parquet"):
            if not path.stem.endswith(suffix):
                continue
            view_name = path.stem[:-len(suffix)].replace("__", "::")
            if view_name.startswith("group_"):
                view_name = "group:" + view_name[len("group_"):]
            available[view_name] = path
        views = list(available) if VIEWS is None else VIEWS
        for view in views:
            if view not in available:
                print(label + f": нет сохранённой схемы {view}, бюджет {budget}; пропуск.")
                continue
            figure = None
            try:
                timeline = pd.read_parquet(available[view])
                figure, alarms = plot_training_timeline(timeline, CONTEXTS[unit], CFG,
                    label, risk, item_status, view, START, END)
                slug = view.replace(":", "_") + suffix
                target = folder/(slug + ".png")
                figure.savefig(target, dpi=130, bbox_inches="tight")
                display(figure)
                show_table("Риск в эпизодах тревоги · " + view, alarms)
                alarms.to_csv(folder/(slug + "_risk_alarms.csv"), index=False, encoding="utf-8-sig")
                FILES.append(dict(experiment=item["experiment"], unit=unit, view=view, png=str(target)))
            except Exception as exc:
                FAILURES.append({"семейство": label, "этап": view, "ошибка": repr(exc)})
                traceback.print_exc()
            finally:
                if figure is not None:
                    plt.close(figure)
        # Важность входов регрессора выводится один раз на эксперимент.
        importance = directory.parent/"feature_importance_summary.csv"
        if importance.exists():
            show_table(label + " · важность входов регрессора (первые 25)", read_report(importance).head(25))

RISK_RESULT = pd.DataFrame(RISK_ROWS)
PLOTS = pd.DataFrame(FILES)
RISK_RESULT.to_csv(REPORT_DIR/"risk_status.csv", index=False, encoding="utf-8-sig")
PLOTS.to_csv(REPORT_DIR/"plots.csv", index=False, encoding="utf-8-sig")
show_table("Итоговые статусы риска", RISK_RESULT)
show_table("Готовые картинки", PLOTS)
if FAILURES:
    show_table("Ошибки, требующие проверки", pd.DataFrame(FAILURES), True)
print("Готово. Файлы:", REPORT_DIR)
print("Рабочие головы риска: output/interpretation_groups; история: output/risk_history.")
print("Регрессоры и их экспорт не переобучались. Для дашборда головы автоматически не присоединялись.")


## Как прочитать результат

- Отдельные графики появятся для каждого доступного котла, `conditions` и `related`.
  Нет котла в каталоге — этот эксперимент не успел сохранить результаты.
- На графике ошибка делится на сохранённый порог: 1 означает границу тревоги.
  Полосы показывают работу, остановы и тревоги; возле тревог — риск по горизонтам.
- В таблице готовности видно, каких аварийных/фоновых независимых событий не хватает
  для обучения либо калибровки. Большое число часов не заменяет несколько независимых аварий.
- Ранние месяцы могут не иметь риска, хотя в конце истории классификатор уже обучился:
  в те месяцы нужные исходы ещё не были известны. Это ожидаемое ограничение честной проверки.
- Классификатор причины («смена режима», «ошибка измерения» и т. п.) здесь не обучается:
  ему нужны подтверждённые человеком метки. Риск аварии получает автоматические метки
  из реестра остановов и наблюдаемого будущего.
- Для повторного просмотра уже рассчитанного задайте `TRAIN_RISK=False`.
  Чтобы ускорить вывод, оставьте `VIEWS=["mean::instant"]`; обе схемы входов сохраняются.
- Не меняйте `risk_horizons` ради ускорения после расчёта, если хотите видеть прежние
  сохранённые оценки: настройка входит в проверку совместимости истории риска.
